# Flatmate RL GRPO 2: Step-by-Step Curriculum

This notebook trains the Flatmate RL broker policy with a step curriculum:

1. collect web-endpoint rollouts from the heuristic policy
2. keep `scenario_id`, `seed`, `step_idx`, `prefix_actions_json`, and `reference_action_json` for every state
3. evaluate expected response vs generated response by step
4. train first on step 1 examples from different seeds
5. expand to steps 1-2, then 1-3, and so on while keeping the same learned adapter

The reward combines:

- JSON/schema validity
- expected-action accuracy for the current step
- live web endpoint transition reward after replaying the stored prefix

This is meant for tracking step-by-step accuracy improvement, not only final episode reward.


In [ ]:
# Restart the kernel after this cell if your notebook runtime asks you to.
%pip install -q "trl>=0.26.2" "transformers>=4.57.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas tqdm


In [ ]:
from __future__ import annotations

import asyncio
import inspect
import json
import math
import random
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import pandas as pd
import torch
import websockets
from datasets import Dataset
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
PREVIOUS_GRPO_DIR = "flatmate-rl-grpo-policy"
PREVIOUS_SFT_DIR = "flatmate-rl-grpo-policy/sft_warmup"
OUTPUT_DIR = "flatmate-rl-grpo-2-step-curriculum"
SFT_OUTPUT_DIR = f"{OUTPUT_DIR}/sft_step_warmup"
RESULTS_DIR = Path(OUTPUT_DIR) / "step_metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS_PER_SCENARIO = 5
START_SEED = 101
MAX_COLLECT_STEPS = 14
MAX_CURRICULUM_STEP = 6  # 1-based. Increase after the loop is stable.
NUM_GENERATIONS = 8
GRPO_STEPS_PER_STAGE = 30
SFT_EPOCHS = 2

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"

SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL


## Web Endpoint Client

The notebook uses the live web endpoint. Every reward replay opens a websocket, resets to the stored scenario/seed, replays the stored prefix actions, then applies the candidate model action.


In [ ]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(
            self.ws_url,
            open_timeout=self.timeout_s,
            ping_timeout=self.timeout_s,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})

async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=START_SEED)
        return {"scenario_id": obs["scenario_id"], "status": obs["status"], "message": obs.get("last_user_message") or obs.get("current_user_request")}

await smoke_test_endpoint()


## Heuristic Policy and Prompts

The heuristic defines the expected response for each state. These expected actions are used for SFT targets, expected-action reward, and step accuracy metrics.


In [ ]:
def trace_tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]

def heuristic_action(obs: dict[str, Any]) -> dict[str, Any] | None:
    if obs.get("done"):
        return None

    tool_names = trace_tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")
    booked = [str(item.get("post_id", "")) for item in obs.get("booked_visits", [])]
    selected_posts = list(obs.get("selected_posts", []))
    last_user_message = str(obs.get("last_user_message", "")).lower()
    buyer_history = obs.get("buyer_conversation_history", [])
    last_role = str(buyer_history[-1].get("role", "")) if buyer_history else ""
    user_has_replied = obs.get("status") == "user_response" and last_role == "user"

    def has(name: str) -> bool:
        return name in tool_names

    def count(name: str) -> int:
        return tool_names.count(name)

    def ask_for_buyer_fields() -> dict[str, Any] | None:
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return None

    def ask_for_seller_fields() -> dict[str, Any] | None:
        need_dietary = "dietary" in remaining
        need_occupation = "occupation_requirement" in remaining
        need_slots = "calendar_slots" in remaining
        if need_dietary and need_occupation and need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        if need_dietary or need_occupation or need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the remaining seller details (dietary setup, who the flat is for, available visit time slots)."}
        return None

    if phase == "seller":
        if not obs.get("seller_profile_stored"):
            ask = ask_for_seller_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("check_table_slot_matches"):
            return {"action_type": "tool_call", "tool_name": "check_table_slot_matches", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("confirm_seller_match"):
            return {"action_type": "tool_call", "tool_name": "confirm_seller_match", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        if not has("offer_matched_listing_to_buyer"):
            return {"action_type": "tool_call", "tool_name": "offer_matched_listing_to_buyer", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        return {"action_type": "tool_call", "tool_name": "schedule_table_visit", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}

    if scenario_id == "task_visit_single_seller_followup" and not obs.get("seller_profile_stored"):
        if not obs.get("buyer_profile_stored"):
            ask = ask_for_buyer_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}
        if not has("search_posts"):
            return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}
        return {"action_type": "tool_call", "tool_name": "close_buyer_conversation", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        ask = ask_for_buyer_fields()
        if ask is not None:
            return ask
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    if not has("search_posts"):
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}

    if scenario_id == "task_visit_single":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "post_023 is available Saturday 11am. Please confirm Saturday 11am if that works."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        return None

    if scenario_id == "task_visit_single_hidden_flex":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "No Tuesday slot matches. I can offer Saturday 1pm or Sunday 5pm instead."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        return None

    if scenario_id == "task_visit_multi":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not selected_posts and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "I shortlisted post_031 at tomorrow 7pm and post_052 at Sunday 4pm. Which listings do you want to pursue?"}
        if "post_031" not in booked and count("contact_poster") == 0 and "confirm tomorrow 7pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm tomorrow 7pm for post_031."}
        if "post_031" not in booked and count("contact_poster") == 0:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        if "post_031" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        if "post_052" not in booked and count("contact_poster") == 1 and "confirm sunday 4pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm Sunday 4pm for post_052."}
        if "post_052" not in booked and count("contact_poster") == 1 and count("book_viewing") == 1:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        if "post_052" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        return None

    return None

def compact_observation(obs: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "buyer_profile_stored": obs.get("buyer_profile_stored", False),
        "seller_profile_stored": obs.get("seller_profile_stored", False),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "selected_posts": obs.get("selected_posts", []),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }

def prompt_from_observation(obs: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Return exactly one JSON action and no extra text.\n\n"
        "Valid action shapes:\n"
        "{\"action_type\":\"assistant_message\",\"assistant_message\":\"...\"}\n"
        "{\"action_type\":\"tool_call\",\"tool_name\":\"...\",\"tool_arguments\":{...}}\n\n"
        f"Observation:\n{json.dumps(compact_observation(obs), ensure_ascii=False, sort_keys=True)}\n\n"
        "Action:\n"
    )


## Collect Step-Labeled Web Rows

With `SEEDS_PER_SCENARIO = 5`, step 1 gets five examples per scenario from different seeds. Later curriculum stages use rows where `step_number <= stage_step`.


In [ ]:
@dataclass
class CurriculumCollectionConfig:
    seeds_per_scenario: int = SEEDS_PER_SCENARIO
    start_seed: int = START_SEED
    max_steps: int = MAX_COLLECT_STEPS

async def collect_curriculum_rows(config: CurriculumCollectionConfig = CurriculumCollectionConfig()) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for scenario_id in SCENARIOS:
        for seed_offset in range(config.seeds_per_scenario):
            seed = config.start_seed + seed_offset
            prefix_actions: list[dict[str, Any]] = []
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                for step_idx in range(config.max_steps):
                    expected = heuristic_action(obs)
                    if expected is None or obs.get("done"):
                        break
                    rows.append({
                        "prompt": prompt_from_observation(obs),
                        "scenario_id": scenario_id,
                        "seed": seed,
                        "episode_key": f"{scenario_id}:{seed}",
                        "step_idx": step_idx,
                        "step_number": step_idx + 1,
                        "prefix_actions_json": json.dumps(prefix_actions, ensure_ascii=False),
                        "reference_action_json": json.dumps(expected, ensure_ascii=False, sort_keys=True),
                    })
                    obs = await env.step(expected)
                    prefix_actions.append(expected)
                    if obs.get("done"):
                        break
            print(f"scenario={scenario_id} seed={seed} total_rows={len(rows)}")
    return rows

rows = await collect_curriculum_rows()
rows_df = pd.DataFrame(rows)
rows_df.to_csv(RESULTS_DIR / "curriculum_rows.csv", index=False)
print(rows_df.groupby(["scenario_id", "step_number"]).size().unstack(fill_value=0))

step1_examples = rows_df[rows_df["step_number"] == 1].sort_values(["scenario_id", "seed"])
display(step1_examples[["scenario_id", "seed", "step_number", "reference_action_json"]].groupby("scenario_id").head(5))

steps_1_to_2_preview = rows_df[rows_df["step_number"] <= 2].sort_values(["scenario_id", "seed", "step_number"])
display(steps_1_to_2_preview[["scenario_id", "seed", "step_number", "reference_action_json"]].head(20))

train_dataset_all = Dataset.from_list([
    {**row, "prompt": [{"role": "user", "content": row["prompt"]}]}
    for row in rows
])
train_dataset_all


## Parsing, Matching, and Metrics

`steps_to_correct_response` means the first generation attempt, from 1 to `attempts`, that exactly matches the expected action. If no attempt matches, it is empty.


In [ ]:
def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get("content", ""))
    return str(completion)

def parse_json_action(text: Any) -> dict[str, Any]:
    text = completion_text(text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("completion does not contain a JSON object")
    action = json.loads(text[start : end + 1])
    if action.get("action_type") == "assistant_message":
        msg = str(action.get("assistant_message", "")).strip()
        if not msg:
            raise ValueError("assistant_message action needs assistant_message")
        return {"action_type": "assistant_message", "assistant_message": msg}
    if action.get("action_type") == "tool_call":
        tool_name = str(action.get("tool_name", "")).strip()
        if not tool_name:
            raise ValueError("tool_call action needs tool_name")
        args = action.get("tool_arguments", {})
        if not isinstance(args, dict):
            raise ValueError("tool_arguments must be an object")
        return {"action_type": "tool_call", "tool_name": tool_name, "tool_arguments": args}
    raise ValueError("action_type must be assistant_message or tool_call")

def normalize_action(action: dict[str, Any]) -> dict[str, Any]:
    if action.get("action_type") == "assistant_message":
        return {
            "action_type": "assistant_message",
            "assistant_message": " ".join(str(action.get("assistant_message", "")).lower().split()),
        }
    return {
        "action_type": "tool_call",
        "tool_name": str(action.get("tool_name", "")),
        "tool_arguments": action.get("tool_arguments", {}) if isinstance(action.get("tool_arguments", {}), dict) else {},
    }

def action_match_parts(got: dict[str, Any], expected: dict[str, Any]) -> dict[str, bool]:
    got_n = normalize_action(got)
    exp_n = normalize_action(expected)
    action_type_match = got_n.get("action_type") == exp_n.get("action_type")
    tool_name_match = action_type_match and got_n.get("action_type") == "tool_call" and got_n.get("tool_name") == exp_n.get("tool_name")
    args_match = tool_name_match and got_n.get("tool_arguments") == exp_n.get("tool_arguments")
    assistant_match = action_type_match and got_n.get("action_type") == "assistant_message" and got_n.get("assistant_message") == exp_n.get("assistant_message")
    exact_match = args_match or assistant_match
    return {
        "exact_match": exact_match,
        "action_type_match": action_type_match,
        "tool_name_match": tool_name_match,
        "args_match": args_match,
        "assistant_match": assistant_match,
    }

def expected_action_score(got: dict[str, Any], expected: dict[str, Any]) -> float:
    parts = action_match_parts(got, expected)
    if parts["exact_match"]:
        return 1.0
    score = 0.0
    if parts["action_type_match"]:
        score += 0.2
    if parts["tool_name_match"]:
        score += 0.5
    if got.get("action_type") == "assistant_message" and expected.get("action_type") == "assistant_message":
        score += 0.2
    return score

def summarize_step_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    if eval_df.empty:
        return eval_df
    return (
        eval_df.groupby(["stage", "scenario_id", "step_number"], as_index=False)
        .agg(
            rows=("exact_match", "size"),
            json_ok_rate=("json_ok", "mean"),
            exact_match_rate=("exact_match", "mean"),
            action_type_match_rate=("action_type_match", "mean"),
            tool_name_match_rate=("tool_name_match", "mean"),
            avg_steps_to_correct=("steps_to_correct_response", "mean"),
            missing_correct_rate=("steps_to_correct_response", lambda s: s.isna().mean()),
        )
        .sort_values(["stage", "scenario_id", "step_number"])
    )


## Reward Functions

The expected-action reward directly optimizes the metrics you asked to track. The endpoint reward keeps the policy grounded in real web-environment consequences.


In [ ]:
REWARD_DEBUG = {"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0}

def json_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        try:
            parse_json_action(completion_text(completion))
            rewards.append(0.5)
            REWARD_DEBUG["valid"] += 1
        except Exception as exc:
            rewards.append(-1.0)
            REWARD_DEBUG["invalid"] += 1
            if len(REWARD_DEBUG["parse_errors"]) < 20:
                REWARD_DEBUG["parse_errors"].append({"error": str(exc), "completion": completion_text(completion)[:240]})
    return rewards

def expected_action_reward(completions, reference_action_json, **kwargs) -> list[float]:
    rewards = []
    for completion, reference_json in zip(completions, reference_action_json):
        try:
            got = parse_json_action(completion_text(completion))
            expected = json.loads(reference_json)
            rewards.append(expected_action_score(got, expected))
        except Exception:
            rewards.append(-0.5)
    return rewards

async def score_one_completion(completion: Any, scenario_id: str, seed: int, prefix_actions_json: str) -> float:
    try:
        action = parse_json_action(completion_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -1.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get("done"):
                    REWARD_DEBUG["prefix_done"] += 1
                    return -0.1

            before_violations = len(obs.get("violations", []))
            before_bookings = len(obs.get("booked_visits", []))
            obs = await env.step(action)

        reward = float(obs.get("reward") or obs.get("step_reward") or 0.0)
        reward += 0.15
        if len(obs.get("violations", [])) > before_violations:
            reward -= 0.75
        if len(obs.get("booked_visits", [])) > before_bookings:
            reward += 1.0
        if obs.get("done") and obs.get("status") != "failed":
            reward += 0.5
        return float(max(-2.0, min(2.0, reward)))
    except Exception as exc:
        if len(REWARD_DEBUG["endpoint_errors"]) < 10:
            REWARD_DEBUG["endpoint_errors"].append(repr(exc))
        return -1.0

async def _score_with_semaphore(semaphore, completion, scenario_id, seed, prefix_actions_json) -> float:
    async with semaphore:
        return await score_one_completion(completion, scenario_id, seed, prefix_actions_json)

async def score_completion_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    # The Space is configured for 4 concurrent envs. Use 3 reward sessions to leave one spare.
    semaphore = asyncio.Semaphore(3)
    tasks = [
        _score_with_semaphore(semaphore, c, s, int(sd), p)
        for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)
    ]
    return list(await asyncio.gather(*tasks))

def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}
    def runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            result["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]

def endpoint_transition_reward(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    return run_async_blocking(score_completion_batch(completions, scenario_id, seed, prefix_actions_json))


## Model Loading

This keeps learning from the prior notebook when a checkpoint exists. Preference order:

1. previous GRPO adapter: `flatmate-rl-grpo-policy`
2. previous SFT adapter: `flatmate-rl-grpo-policy/sft_warmup`
3. base Qwen model


In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_merged_start_model():
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
    for adapter_dir in [PREVIOUS_GRPO_DIR, PREVIOUS_SFT_DIR]:
        adapter_path = Path(adapter_dir)
        if (adapter_path / "adapter_config.json").exists():
            print("Starting from adapter:", adapter_dir)
            model = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
            if hasattr(model, "peft_config"):
                delattr(model, "peft_config")
            return model
    print("Starting from base model:", MODEL_NAME)
    return base


## SFT Step Warmup

This optional warmup teaches the model to emit the expected action format for the collected step examples before GRPO starts optimizing rewards.


In [ ]:
def make_sft_config(**kwargs):
    valid = set(inspect.signature(SFTConfig.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported SFTConfig fields:", skipped)
    return SFTConfig(**filtered)

def make_sft_trainer(**kwargs):
    valid = set(inspect.signature(SFTTrainer.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported SFTTrainer fields:", skipped)
    return SFTTrainer(**filtered)

sft_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["reference_action_json"]}],
    }
    for row in rows
])

sft_args = make_sft_config(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=2,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    max_length=1700,
    completion_only_loss=True,
    report_to="none",
)

sft_start_model = load_merged_start_model()
sft_trainer = make_sft_trainer(
    model=sft_start_model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    tokenizer=tokenizer,
    peft_config=peft_config,
)
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
print("Saved step SFT warmup to", SFT_OUTPUT_DIR)


## Evaluation: Expected vs Got

Run this before and after each curriculum stage. It records the expected action, generated action, parse status, match fields, and `steps_to_correct_response`.


In [ ]:
def load_adapter_for_eval(adapter_dir: str):
    model = AutoPeftModelForCausalLM.from_pretrained(adapter_dir, device_map="auto")
    model.eval()
    model.config.use_cache = False
    return model

def build_inputs(prompt: str, model):
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return tokenizer(chat_text, return_tensors="pt", truncation=True, max_length=1700).to(model.device)

def generate_once(model, prompt: str, sample: bool, temperature: float = 0.7) -> str:
    inputs = build_inputs(prompt, model)
    generation_kwargs = {
        "max_new_tokens": 200,
        "do_sample": sample,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if sample:
        generation_kwargs.update({"temperature": temperature, "top_p": 0.95})
    with torch.no_grad():
        output = model.generate(**inputs, **generation_kwargs)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def evaluate_expected_vs_got(model, eval_rows: list[dict[str, Any]], stage: str, attempts: int = 5) -> pd.DataFrame:
    records = []
    progress = tqdm(eval_rows, desc=f"eval {stage}", total=len(eval_rows), dynamic_ncols=True)
    for row in progress:
        progress.set_postfix({"scenario": row["scenario_id"], "seed": row["seed"], "step": row["step_number"]})
        expected = json.loads(row["reference_action_json"])
        first_raw = ""
        first_error = ""
        first_got: dict[str, Any] | None = None
        first_parts = {"exact_match": False, "action_type_match": False, "tool_name_match": False, "args_match": False, "assistant_match": False}
        steps_to_correct = None

        for attempt_idx in range(1, attempts + 1):
            raw = generate_once(model, row["prompt"], sample=(attempt_idx > 1))
            if attempt_idx == 1:
                first_raw = raw
            try:
                got = parse_json_action(raw)
                parts = action_match_parts(got, expected)
                if attempt_idx == 1:
                    first_got = got
                    first_parts = parts
                if parts["exact_match"] and steps_to_correct is None:
                    steps_to_correct = attempt_idx
                    break
            except Exception as exc:
                if attempt_idx == 1:
                    first_error = str(exc)

        records.append({
            "stage": stage,
            "scenario_id": row["scenario_id"],
            "seed": row["seed"],
            "episode_key": row["episode_key"],
            "step_number": row["step_number"],
            "expected_action_json": row["reference_action_json"],
            "got_action_json": json.dumps(first_got, ensure_ascii=False, sort_keys=True) if first_got else "",
            "raw_completion": first_raw,
            "json_ok": first_got is not None,
            "parse_error": first_error,
            "steps_to_correct_response": steps_to_correct,
            **first_parts,
        })
    return pd.DataFrame(records)

# Evaluate the SFT warmup before GRPO curriculum.
eval_model = load_adapter_for_eval(SFT_OUTPUT_DIR)
baseline_eval_df = evaluate_expected_vs_got(eval_model, rows, stage="sft_warmup", attempts=5)
baseline_eval_df.to_csv(RESULTS_DIR / "eval_sft_warmup.csv", index=False)
baseline_summary = summarize_step_metrics(baseline_eval_df)
baseline_summary.to_csv(RESULTS_DIR / "summary_sft_warmup.csv", index=False)
baseline_summary.head(20)


## Curriculum GRPO

Stage 1 trains on only `step_number <= 1`. Stage 2 trains on `step_number <= 2`, and so on. The adapter from the previous stage becomes the starting point for the next stage.


In [ ]:
def make_grpo_config(**kwargs):
    valid = set(inspect.signature(GRPOConfig.__init__).parameters)
    filtered = {k: v for k, v in kwargs.items() if k in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported GRPOConfig fields:", skipped)
    return GRPOConfig(**filtered)

def dataset_for_stage(max_step_number: int) -> Dataset:
    stage_rows = [row for row in rows if int(row["step_number"]) <= int(max_step_number)]
    return Dataset.from_list([
        {**row, "prompt": [{"role": "user", "content": row["prompt"]}]}
        for row in stage_rows
    ])

def load_stage_start_model(adapter_dir: str):
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
    if hasattr(model, "peft_config"):
        delattr(model, "peft_config")
    return model

all_eval_dfs = [baseline_eval_df]
all_summaries = [baseline_summary]
current_adapter_dir = SFT_OUTPUT_DIR

for stage_step in range(1, MAX_CURRICULUM_STEP + 1):
    stage_name = f"steps_1_to_{stage_step}"
    stage_output_dir = f"{OUTPUT_DIR}/{stage_name}"
    print("\n=== GRPO curriculum stage:", stage_name, "===")

    stage_dataset = dataset_for_stage(stage_step)
    print("stage rows:", len(stage_dataset))

    stage_model = load_stage_start_model(current_adapter_dir)
    grpo_args = make_grpo_config(
        output_dir=stage_output_dir,
        learning_rate=5e-6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=1700,
        max_completion_length=200,
        max_steps=GRPO_STEPS_PER_STAGE,
        logging_steps=1,
        save_steps=max(10, GRPO_STEPS_PER_STAGE),
        save_total_limit=1,
        beta=0.1,
        temperature=1.0,
        top_p=0.95,
        report_to="none",
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
    )

    REWARD_DEBUG.update({"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0})
    trainer = GRPOTrainer(
        model=stage_model,
        processing_class=tokenizer,
        reward_funcs=[json_format_reward, expected_action_reward, endpoint_transition_reward],
        args=grpo_args,
        train_dataset=stage_dataset,
        peft_config=peft_config,
    )
    train_result = trainer.train()
    trainer.save_model(stage_output_dir)
    current_adapter_dir = stage_output_dir

    print("Reward debug:", REWARD_DEBUG)
    (Path(stage_output_dir) / "train_log_history.json").write_text(json.dumps(trainer.state.log_history, indent=2))

    eval_model = load_adapter_for_eval(stage_output_dir)
    eval_df = evaluate_expected_vs_got(eval_model, rows, stage=stage_name, attempts=5)
    summary_df = summarize_step_metrics(eval_df)
    eval_df.to_csv(RESULTS_DIR / f"eval_{stage_name}.csv", index=False)
    summary_df.to_csv(RESULTS_DIR / f"summary_{stage_name}.csv", index=False)
    all_eval_dfs.append(eval_df)
    all_summaries.append(summary_df)

    display(summary_df[summary_df["step_number"] <= stage_step].head(40))

combined_eval_df = pd.concat(all_eval_dfs, ignore_index=True)
combined_summary_df = pd.concat(all_summaries, ignore_index=True)
combined_eval_df.to_csv(RESULTS_DIR / "eval_all_stages.csv", index=False)
combined_summary_df.to_csv(RESULTS_DIR / "summary_all_stages.csv", index=False)
combined_summary_df.tail(30)


## Accuracy Plots

These plots show whether each stage improves exact-match accuracy and reduces attempts needed to produce the correct response.


In [ ]:
import matplotlib.pyplot as plt

plot_df = combined_summary_df.copy()
plot_df["stage_order"] = plot_df["stage"].map({stage: idx for idx, stage in enumerate(plot_df["stage"].drop_duplicates())})

for scenario_id in SCENARIOS:
    sub = plot_df[plot_df["scenario_id"] == scenario_id]
    if sub.empty:
        continue
    pivot = sub.pivot_table(index="stage", columns="step_number", values="exact_match_rate", aggfunc="mean")
    ax = pivot.plot(marker="o", figsize=(10, 4), title=f"Exact match by step: {scenario_id}")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("exact match rate")
    plt.show()

attempts_pivot = combined_summary_df.pivot_table(index="stage", columns="step_number", values="avg_steps_to_correct", aggfunc="mean")
ax = attempts_pivot.plot(marker="o", figsize=(10, 4), title="Average attempts to correct response")
ax.grid(True, alpha=0.3)
ax.set_ylabel("attempts; lower is better")
plt.show()


## Inspect Mistakes

Use this to see expected response vs got response for any stage and step.


In [ ]:
def show_mistakes(stage: str, step_number: int, limit: int = 20):
    df = combined_eval_df[
        (combined_eval_df["stage"] == stage)
        & (combined_eval_df["step_number"] == step_number)
        & (~combined_eval_df["exact_match"])
    ].copy()
    cols = [
        "scenario_id",
        "seed",
        "step_number",
        "steps_to_correct_response",
        "expected_action_json",
        "got_action_json",
        "parse_error",
        "raw_completion",
    ]
    return df[cols].head(limit)

show_mistakes(stage=f"steps_1_to_{MAX_CURRICULUM_STEP}", step_number=1, limit=10)
